# 02 Preprocessing

This notebook implements the data ingestion and preprocessing pipeline for the project. It reads raw assets from `data/raw/`, applies cleaning and schema enforcement, and writes cleaned output to `data/processed/`.

## Objectives
- Load raw game log assets
- Validate and clean columns
- Standardize timestamp and numeric fields
- Add derived fields for analysis
- Export a cleaned sample dataset

In [1]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path('../data/raw/player_game_logs_raw.csv').resolve()
OUTPUT_PATH = Path('../data/processed/player_game_logs_processed.csv').resolve()

print('Raw path:', RAW_PATH)
print('Output path:', OUTPUT_PATH)

Raw path: C:\Users\ogund\Downloads\lerol\Neural-Domination-Group-Project\data\raw\player_game_logs_raw.csv
Output path: C:\Users\ogund\Downloads\lerol\Neural-Domination-Group-Project\data\processed\player_game_logs_processed.csv


In [2]:
df = pd.read_csv(RAW_PATH)
print('Raw rows:', len(df))
print(df.dtypes)
df.head()

Raw rows: 6
game_id           int64
timestamp           str
player_id           str
action              str
target              str
score_change      int64
position_x      float64
position_y      float64
raw_note            str
dtype: object


,game_id,timestamp,player_id,action,target,score_change,position_x,position_y,raw_note
0,1,2026-04-01 09:12:05,P001,move,NaN,0,12.4,5.7,started session
1,1,2026-04-01 09:12:18,P001,attack,enemy_1,15,13.0,6.2,first attack
2,1,2026-04-01 09:12:42,P002,move,NaN,0,7.8,8.4,position update
3,1,2026-04-01 09:13:00,P002,defend,NaN,5,8.0,8.0,blocked hit
4,1,2026-04-01 09:13:15,P001,heal,P002,10,12.6,5.9,healed teammate


## Schema enforcement and cleaning steps
1. Rename and verify required columns.
2. Parse `timestamp` as datetime.
3. Fill missing scores and targets.
4. Drop rows missing `player_id` or `action` values.
5. Normalize position values to support model input.
6. Add a binary `is_combat_action` feature.

In [3]:
required_columns = [
    'game_id',
    'timestamp',
    'player_id',
    'action',
    'target',
    'score_change',
    'position_x',
    'position_y'
]
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df = df[required_columns].copy()
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['player_id'] = df['player_id'].astype('string').str.strip()
df['action'] = df['action'].astype('string').str.lower().str.strip()
df['target'] = df['target'].astype('string').replace({'nan': ''}).replace('None', '')
df['score_change'] = pd.to_numeric(df['score_change'], errors='coerce').fillna(0).astype(int)
df['position_x'] = pd.to_numeric(df['position_x'], errors='coerce')
df['position_y'] = pd.to_numeric(df['position_y'], errors='coerce')

df = df.dropna(subset=['player_id', 'action', 'timestamp'])

df['position_x'] = df['position_x'].fillna(df['position_x'].median())
df['position_y'] = df['position_y'].fillna(df['position_y'].median())

# Derive normalized coordinates assuming field range [0, 20] for x and [0, 30] for y
df['normalized_x'] = (df['position_x'] / 20).clip(0, 1).round(2)
df['normalized_y'] = (df['position_y'] / 30).clip(0, 1).round(2)

combat_actions = {'attack', 'defend', 'heal'}
df['is_combat_action'] = df['action'].isin(combat_actions)

df = df.sort_values(['game_id', 'timestamp']).reset_index(drop=True)
df.head()

,game_id,timestamp,player_id,action,target,score_change,position_x,position_y,normalized_x,normalized_y,is_combat_action
0,1,2026-04-01 09:12:05,P001,move,<NA>,0,12.4,5.7,0.62,0.19,False
1,1,2026-04-01 09:12:18,P001,attack,enemy_1,15,13.0,6.2,0.65,0.21,True
2,1,2026-04-01 09:12:42,P002,move,<NA>,0,7.8,8.4,0.39,0.28,False
3,1,2026-04-01 09:13:00,P002,defend,<NA>,5,8.0,8.0,0.40,0.27,True
4,1,2026-04-01 09:13:15,P001,heal,P002,10,12.6,5.9,0.63,0.20,True


## Export cleaned sample output
The processed file includes a stable schema and derived fields helpful for downstream training or analysis.

In [4]:
df.to_csv(OUTPUT_PATH, index=False)
print('Saved cleaned data to', OUTPUT_PATH)

Saved cleaned data to C:\Users\ogund\Downloads\lerol\Neural-Domination-Group-Project\data\processed\player_game_logs_processed.csv


## Notes and assumptions
- `game_id` is treated as a categorical session identifier.
- Missing target values are represented as empty strings.
- `score_change` uses 0 for missing or invalid values.
- Position coordinates are normalized assuming a fixed playfield area.
- Combat actions are defined as `attack`, `defend`, and `heal`.